# 2 — Plotting: the same chart, ten times

A first plot is never the final plot. The skill worth learning is not "how do I
draw a line" — it is **how do I look at a mediocre chart and know what to change
next.**

So this notebook draws one figure and then revises it, one change per cell, until
it is something you would put in front of a professor or a portfolio manager.
Then it does the other chart types you actually need, and finishes with a figure
you build yourself from a checklist.

* Exercises are marked **Your turn** and graded with `check('2.1', ...)`.
* `hint('2.1')` if you are stuck.
* The checker looks *inside* your figure — it can tell whether your axes have
  labels and whether your legend has real names in it.

**Assumed:** notebook 1. You should be comfortable with `pd.read_csv`, filtering,
and `pivot`.

**Not in here:** no statistics, no models. Just drawing.

In [ ]:
# Run me first. This makes workbook.py importable whatever folder Jupyter
# started in, then pulls in the three helpers you will use all the way through.
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'Learn_To_Code', *Path.cwd().parents]:
    if (candidate / 'workbook.py').exists():
        sys.path.insert(0, str(candidate))
        break

from workbook import check, hint, todo, ensure_data, DATA_DIR

ensure_data()   # builds the workbook CSVs on your Desktop the first time only

print('Ready. Data lives in:', DATA_DIR)

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

prices = pd.read_csv(DATA_DIR / 'stock_prices.csv', parse_dates=['date'])

# One column per ticker — the shape almost every plotting call wants.
wide = prices.pivot(index='date', columns='ticker', values='price')
wide.head()

`%matplotlib inline` tells Jupyter to draw figures underneath the cell that made
them. It is the default in most setups; writing it out means the notebook behaves
the same on someone else's machine.

## 1. Two ways to plot, and why you should only use one

pandas will plot a DataFrame directly:

```python
wide.plot()
```

That is genuinely useful for a ten-second look at your data. It is not useful for
anything you intend to show another human, because it hands you no handle on the
result.

The other way gives you that handle:

```python
fig, ax = plt.subplots()
ax.plot(x, y)
```

* `fig` is the **Figure** — the piece of paper. It owns the size, the resolution,
  and the file you save.
* `ax` is the **Axes** — one set of x and y axes drawn on that paper. Everything
  you actually care about (lines, labels, ticks, legend, limits) belongs to an
  Axes.

Almost every plotting problem you will hit has the same answer: **keep `ax` and
call a method on it.**

In [ ]:
# The quick look. Fine for you; not fine for anyone else.
wide.plot(figsize=(9, 4))
plt.show()

In [ ]:
# The same picture, drawn the way we will do it from here on.
fig, ax = plt.subplots(figsize=(9, 4))
for ticker in wide.columns:
    ax.plot(wide.index, wide[ticker])
plt.show()

### The parts of a figure

You will be told to "fix the ticks" or "drop the spines". This is what those
words point at.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.linspace(0, 10, 200)
ax.plot(x, np.sin(x) + 3.0, color='#0173B2', linewidth=2, label='a line (an "artist")')
ax.set_xlabel('x axis label')
ax.set_ylabel('y axis label')
ax.set_title('Title')
ax.legend(loc='lower right')
ax.set_ylim(0, 9)

arrow = dict(arrowstyle='->', color='crimson', linewidth=1.2)
label = dict(color='crimson', fontsize=10, xycoords='axes fraction',
             textcoords='axes fraction', arrowprops=arrow)

ax.annotate('the Axes — everything below belongs to it',
            xy=(0.35, 0.62), xytext=(0.22, 0.90), **label)
ax.annotate('spine (there are four)',
            xy=(1.00, 0.52), xytext=(0.97, 0.72), ha='right', **label)
ax.annotate('tick and tick label',
            xy=(0.20, 0.00), xytext=(0.10, 0.17), **label)
ax.annotate('the legend',
            xy=(0.88, 0.09), xytext=(0.60, 0.20), ha='right', **label)

fig.text(0.01, 0.97, 'the Figure is this whole white sheet',
         color='crimson', fontsize=10)
plt.show()

## 2. The revision ladder

The question the chart has to answer: **how did these three stocks do over the
sample?** Watch each version and ask what is still stopping a reader from
answering it.

### v1 — the default

In [ ]:
three = ['ACME', 'CRUX', 'EVER']

fig, ax = plt.subplots()
for t in three:
    ax.plot(wide.index, wide[t])
plt.show()

Three unnamed lines, no units, no title. A reader cannot tell which line is which
or what the numbers mean.

### v2 — say what the axes are

Never show anyone a chart whose axes have no names. The unit belongs in the
label: `Price ($)`, not `Price`.

In [ ]:
fig, ax = plt.subplots()
for t in three:
    ax.plot(wide.index, wide[t])

ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.set_title('Three stocks, daily close')
plt.show()

### Your turn

In [ ]:
# Plot just BOLT, and give the figure an x label, a y label with the unit in it,
# and a title. Three method calls on ax.

fig, ax = plt.subplots()
ax.plot(wide.index, wide['BOLT'])

# your labels here

plt.show()
check('2.1', ax)

### v3 — plot the right number

The single most common fix for a bad chart is not a styling change: it is
plotting something else. EVER starts near \$205 and FLUX near \$27, so on a price
axis the cheap stock is a flat line at the bottom regardless of how it did.

Rebase every series to 100 at the start and the chart finally answers the
question it was asked.

In [ ]:
rebased = wide / wide.iloc[0] * 100.0

fig, ax = plt.subplots()
for t in three:
    ax.plot(rebased.index, rebased[t])

ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested at the start')
ax.set_title('Three stocks, rebased to 100')
plt.show()

### v4 — name the lines, and choose the colours

`label=` inside each `plot` call, then one `ax.legend()` at the end. Without a
label a line shows up in the legend as `_child0`, which is matplotlib telling you
it had nothing to work with.

While you are there, take control of colour. The defaults are fine, but colour is
information: keep one series one colour across every chart in a deck, and pick
something colour-blind readers can separate (about 1 man in 12 cannot reliably
tell red from green).

In [ ]:
colours = {'ACME': '#0173B2', 'CRUX': '#DE8F05', 'EVER': '#029E73'}

fig, ax = plt.subplots()
for t in three:
    ax.plot(rebased.index, rebased[t],
            label=t, color=colours[t], linewidth=1.6)

ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested at the start')
ax.set_title('Three stocks, rebased to 100')
ax.legend(frameon=False)
plt.show()

The knobs you will use over and over:

| Argument     | Does | Typical values |
| ------------ | ---- | -------------- |
| `color`      | line colour | `'crimson'`, `'#0173B2'`, `'C0'` |
| `linewidth`  | thickness | `1.0`–`2.5` |
| `linestyle`  | dashing | `'-'`, `'--'`, `':'`, `'-.'` |
| `alpha`      | transparency | `0.3` for background series |
| `marker`     | point markers | `'o'`, `'.'`, `'s'` — only when points are few |
| `zorder`     | what draws on top | higher is nearer the front |

**Dash the series that are context, keep solid for the series that is the point.**
That, plus alpha, replaces most legends.

### Your turn

In [ ]:
# Plot ACME, BOLT and DYNE from `rebased` on one Axes.
# Each line needs its own label=, and the figure needs a legend.

fig, ax = plt.subplots()

# your three plot calls, then ax.legend()

plt.show()
check('2.2', ax)

### v5 — control the axes

Four things you will adjust constantly:

* `ax.set_xlim` / `ax.set_ylim` — the window. Cutting a series off mid-move is a
  real distortion, so change limits deliberately, not to make a result look better.
* `ax.set_yscale('log')` — on a log axis **equal vertical distances are equal
  percentage moves**, which is what you want whenever you are comparing growth.
* `ax.tick_params` — size and direction of ticks.
* date formatting, via `matplotlib.dates`.

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots()
for t in three:
    ax.plot(rebased.index, rebased[t],
            label=t, color=colours[t], linewidth=1.6)

ax.set_yscale('log')
ax.set_yticks([50, 75, 100, 150, 200, 300])
ax.yaxis.set_major_formatter(plt.ScalarFormatter())   # 100, not 10^2
ax.yaxis.set_minor_formatter(plt.NullFormatter())     # and no stray 6x10^1

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested (log scale)')
ax.set_title('Three stocks, rebased to 100')
ax.legend(frameon=False)
plt.show()

### Your turn

In [ ]:
# Plot ACME from `wide` and put the y axis on a log scale.

fig, ax = plt.subplots()
ax.plot(wide.index, wide['ACME'])
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')

# your one line here

plt.show()
check('2.3', ax)

### v6 — take things away

Every element that is not carrying information is competing with the ones that
are. Drop the top and right spines, put the grid behind the data and make it
faint, and let the lines be the darkest thing on the page.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for t in three:
    ax.plot(rebased.index, rebased[t],
            label=t, color=colours[t], linewidth=1.8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linewidth=0.8)
ax.set_axisbelow(True)                 # grid behind the lines, not through them
ax.tick_params(direction='out', length=4, labelsize=10)

ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested at the start')
ax.set_title('Three stocks, rebased to 100')
ax.legend(frameon=False, loc='upper left')
plt.show()

### v7 — point at the thing you want them to see

A chart that makes the reader hunt for the finding has not finished. Annotate it.

* `ax.annotate(text, xy=..., xytext=..., arrowprops=...)` — a labelled arrow.
* `ax.axhline` / `ax.axvline` — a reference level or date.
* `ax.axvspan` / `ax.axhspan` — shade a period or a band.
* `ax.text` — plain text at a position.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for t in three:
    ax.plot(rebased.index, rebased[t],
            label=t, color=colours[t], linewidth=1.8)

ax.axhline(100, color='0.5', linewidth=1, linestyle='--')   # you break even here

# Shade the stretch we want to talk about — found from the data, not typed in
trough = rebased['CRUX'].idxmin()
peak = rebased['CRUX'].loc[:trough].idxmax()
ax.axvspan(peak, trough, color='0.88', zorder=0)
ax.text(peak + (trough - peak) / 2, ax.get_ylim()[1] * 0.95,
        'CRUX peak to trough', ha='center', fontsize=10, color='0.35')

# Put an arrow on the single fact the chart exists to show.
# textcoords='offset points' positions the text relative to the arrow tip,
# so it can never wander off the axes.
ax.annotate(f'bottoms at {rebased["CRUX"].min():.0f}',
            xy=(trough, rebased['CRUX'].min()),
            xytext=(-30, 40), textcoords='offset points',
            ha='right', fontsize=10,
            arrowprops=dict(arrowstyle='->', color='0.3'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested at the start')
ax.set_title('CRUX never recovered; ACME and EVER did')
ax.legend(frameon=False, loc='upper left')
plt.show()

Notice the title changed. **A title that states the finding is worth more than a
title that names the data.** "Three stocks, rebased to 100" is a caption; "CRUX
never recovered" is a point.

That is the ladder. Same three lines throughout — every improvement came from
deciding what the reader needed next.

## 3. The other charts you need

Four types cover the overwhelming majority of analytical work.

| You want to show | Use | Method |
| ---------------- | --- | ------ |
| something over time | line | `ax.plot` |
| one variable against another | scatter | `ax.scatter` |
| the shape of one variable | histogram | `ax.hist` |
| a value per category | bar | `ax.bar` / `ax.barh` |

If you are reaching for a pie chart, use a bar chart. People read length far more
accurately than angle.

### Scatter — risk against return

In [ ]:
daily = wide.pct_change().dropna()

summary = pd.DataFrame({
    'ann_vol': daily.std() * np.sqrt(252) * 100,
    'ann_ret': ((wide.iloc[-1] / wide.iloc[0]) ** (1 / 3) - 1) * 100,
})
summary.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(summary['ann_vol'], summary['ann_ret'],
           s=90, color='#0173B2', zorder=3)

for name, row in summary.iterrows():
    ax.annotate(name, (row['ann_vol'], row['ann_ret']),
                xytext=(6, 4), textcoords='offset points', fontsize=10)

ax.axhline(0, color='0.6', linewidth=1)
ax.set_xlabel('Annualised volatility (%)')
ax.set_ylabel('Annualised return (%)')
ax.set_title('Six stocks: what did the risk buy?')
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
plt.show()

With six points, labelling every one beats a legend. With six hundred, label the
handful that matter and leave the rest anonymous.

### Your turn

In [ ]:
# Scatter of average daily volume (x) against annualised volatility (y),
# one point per ticker, with every point labelled and both axes named.

vol_summary = pd.DataFrame({
    'avg_volume': prices.groupby('ticker')['volume'].mean() / 1e6,
    'ann_vol': daily.std() * np.sqrt(252) * 100,
})

fig, ax = plt.subplots(figsize=(7, 5))

# your ax.scatter, your annotate loop, your labels

plt.show()
check('2.4', ax)

### Histogram — the shape of daily returns

In [ ]:
r = daily['ACME'] * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(r, bins=60, color='#0173B2', alpha=0.85)

ax.axvline(r.mean(), color='crimson', linestyle='--', linewidth=1.5,
           label=f'mean {r.mean():.2f}%')
ax.axvline(0, color='0.4', linewidth=1)

ax.set_xlabel('Daily return (%)')
ax.set_ylabel('Number of days')
ax.set_title('ACME daily returns')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

`bins` is the only argument that really matters, and it changes the story: too
few and you flatten everything interesting, too many and you are looking at
noise. Try 10 and 200 in the cell above before you move on — deciding the bin
count *is* the analysis.

### Your turn

In [ ]:
# Histogram of DYNE's daily returns in percent, with a dashed vertical line
# at the mean. Use plenty of bins.

dyne = daily['DYNE'] * 100

fig, ax = plt.subplots(figsize=(8, 4.5))

# your ax.hist and your dashed ax.axvline

plt.show()
check('2.5', ax)

### Bar — one value per category

`barh` (horizontal) is usually the better choice: category names read left to
right, so they never have to be rotated. Sort the bars unless the categories have
a natural order of their own.

In [ ]:
total_return = (wide.iloc[-1] / wide.iloc[0] - 1).sort_values() * 100

fig, ax = plt.subplots(figsize=(7, 4))
bar_colours = ['#C44E52' if v < 0 else '#0173B2' for v in total_return]
ax.barh(total_return.index, total_return.values, color=bar_colours)

for name, v in total_return.items():
    ax.text(v + (2 if v >= 0 else -2), name, f'{v:.0f}%',
            va='center', ha='left' if v >= 0 else 'right', fontsize=10)

ax.axvline(0, color='0.3', linewidth=1)
ax.set_xlabel('Total return over the sample (%)')
ax.set_title('Three winners, two losers, one flat')
ax.set_xlim(-60, 140)
for side in ['top', 'right', 'left']:
    ax.spines[side].set_visible(False)
plt.show()

## 4. Small multiples

When six series on one Axes become spaghetti, give each its own panel. The rule
that makes small multiples work: **every panel shares the same axes**, so the eye
can compare them directly. `sharex` and `sharey` do that, and they remove the
duplicated tick labels for free.

`plt.subplots(2, 3)` returns an array of Axes. `.flat` lets you loop over them in
one line.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)

for ax, t in zip(axes.flat, wide.columns):
    ax.plot(rebased.index, rebased[t], color='#0173B2', linewidth=1.4)
    ax.axhline(100, color='0.6', linewidth=0.8, linestyle='--')
    ax.set_title(t, fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # Narrow panels need fewer ticks, or the dates collide into a smear.
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Growth of 100, each stock on the same axes', fontsize=13)
fig.supylabel('Value of 100')
fig.tight_layout()
plt.show()

`fig.tight_layout()` fixes overlapping labels. Call it once, last. If a title is
still clipped after that, the figure is too small — grow `figsize` rather than
shrinking the font.

### Your turn

In [ ]:
# A 2x2 grid. One ticker per panel: ACME, BOLT, CRUX, DYNE.
# Plot each one's price from `wide`, and give every panel a title.

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)

# your loop over zip(axes.flat, ['ACME', 'BOLT', 'CRUX', 'DYNE'])

fig.tight_layout()
plt.show()
check('2.6', fig)

## 5. Twin axes — and a warning

`ax.twinx()` puts a second y scale on the right. Use it when two series genuinely
belong in one picture but have incompatible units — a price and a volume, say.

Be careful: with two free scales you can make any two series look related or
unrelated just by choosing the limits. If both series share a unit, do not use a
twin axis — rebase them and use one.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(wide.index, wide['ACME'], color='#0173B2', linewidth=1.6, label='Price')
ax.set_ylabel('Price ($)', color='#0173B2')
ax.tick_params(axis='y', labelcolor='#0173B2')

ax2 = ax.twinx()
volume = prices[prices['ticker'] == 'ACME'].set_index('date')['volume']
ax2.bar(volume.index, volume / 1e6, color='0.75', width=1.0, zorder=0)
ax2.set_ylabel('Volume (millions)', color='0.5')
ax2.tick_params(axis='y', labelcolor='0.5')
ax2.set_ylim(0, 8)      # leaves the top half of the panel for the price line

ax.set_zorder(ax2.get_zorder() + 1)    # line in front of bars
ax.patch.set_visible(False)            # ...and let the bars show through
ax.set_title('ACME price and volume')
plt.show()

## 6. Setting a house style once

Retyping `spines[...].set_visible(False)` on every figure gets old. `rcParams` are
matplotlib's defaults: set them once near the top of a notebook and every figure
afterwards inherits them.

In [ ]:
plt.rcParams.update({
    'figure.figsize': (9, 5),
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'legend.frameon': False,
    'lines.linewidth': 1.8,
    'font.size': 11,
})

# Same six lines of plotting code as v1 — everything else came from rcParams.
fig, ax = plt.subplots()
for t in three:
    ax.plot(rebased.index, rebased[t], label=t)
ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested')
ax.set_title('House style, for free')
ax.legend()
plt.show()

Ready-made styles exist too — `plt.style.use('seaborn-v0_8-whitegrid')`,
`'ggplot'`, `'fivethirtyeight'`; `plt.style.available` lists them. A style sheet
is a shortcut for the same `rcParams`.

**A note on colour.** `'C0'` … `'C9'` are the current cycle, so your colours stay
consistent if you change style later. For categorical series pick a
colour-blind-safe set; for a quantity that runs low-to-high use a sequential
colormap (`viridis`), and for something centred on zero use a diverging one
(`RdBu`) — never a rainbow, which invents boundaries that are not in the data.

## 7. Saving a figure properly

```python
fig.savefig(path, dpi=150, bbox_inches='tight')
```

Three things that go wrong:

1. **`plt.savefig()` after `plt.show()` writes a blank file.** Showing the figure
   clears it. Save first, or save from the `fig` object as above — `fig.savefig`
   always knows which figure it means.
2. **Default dpi is too low for print.** 150 for a document, 300 for a paper.
3. **Labels get cut off.** `bbox_inches='tight'` crops to the content instead of
   the nominal figure box.

Use `.png` for anything you paste into slides, `.pdf` or `.svg` for anything that
goes into LaTeX or gets scaled up — those stay sharp at any size.

In [ ]:
from pathlib import Path

desktop = Path.home() / 'Desktop'
out_dir = desktop if desktop.is_dir() else Path.home()

fig, ax = plt.subplots()
for t in three:
    ax.plot(rebased.index, rebased[t], label=t, color=colours[t])
ax.set_xlabel('Date')
ax.set_ylabel('Value of 100 invested')
ax.set_title('Saved from the fig object')
ax.legend()

saved_to = out_dir / 'bootcamp_demo_figure.png'
fig.savefig(saved_to, dpi=150, bbox_inches='tight')
plt.show()

print('written to', saved_to)
print('size', saved_to.stat().st_size // 1024, 'KB')

## 8. Your turn: build one properly

No scaffolding this time. Build a figure that shows the growth of 100 in **ACME,
DYNE and EVER**, and meets all of this:

- [ ] created with `plt.subplots`, at least 8 inches wide
- [ ] three lines, each with its own `label=`
- [ ] a legend
- [ ] both axes labelled, with units
- [ ] a title that says something, not just names the data
- [ ] saved to your Desktop as `my_first_figure.png` at `dpi=150`

`check('2.7', fig, figure_path)` inspects the figure *and* the file on disk.

In [ ]:
figure_path = out_dir / 'my_first_figure.png'
picks = ['ACME', 'DYNE', 'EVER']

fig, ax = plt.subplots()      # give it a figsize

# your code here

check('2.7', fig, figure_path)

## 9. The checklist to run in your head

Before any chart leaves your machine:

1. Can a reader say what is being measured, in what unit, over what period —
   without asking you?
2. Is every line identified, by a legend or a direct label?
3. Does the title state the finding?
4. Is the y axis doing something honest? (Truncated axes and double axes are
   where charts lie.)
5. Is anything on there that carries no information? Remove it.
6. Does it survive being printed in black and white?
7. Is it saved at a resolution someone can actually read?

### Things that will bite you

| Symptom | Cause |
| ------- | ----- |
| saved image is blank | `plt.savefig` after `plt.show()` |
| legend says `_child0` | no `label=` in the plot call |
| labels cut off in the file | missing `bbox_inches='tight'` |
| panels overlap | missing `fig.tight_layout()` |
| figure appears twice | a stray `fig` on the last line as well as `plt.show()` |
| `RuntimeWarning: More than 20 figures` | making figures in a loop without `plt.close(fig)` |

### Next

**Notebook 3 — Regression.** You have looked at the data; now fit something to
it, first with matrix algebra you write yourself and then with the packages
everyone uses.